In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
import warnings
import joblib

In [ ]:
df = pd.read_csv("train (1).csv")

In [ ]:
df.isnull().count()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df = df.drop('PoolQC', axis=1)
df = df.drop('MiscFeature', axis=1)
df = df.drop('Alley', axis=1)
df = df.drop('Fence', axis=1)

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df = df.drop('MasVnrType', axis=1)

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df["FireplaceQu"] = df["FireplaceQu"].fillna("No fireplace")

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].mean())

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df["GarageType"] = df["GarageType"].fillna("No Garage")
df["GarageCond"] = df["GarageCond"].fillna("No Garage")
df["GarageQual"] = df["GarageQual"].fillna("No Garage")
df["GarageFinish"] = df["GarageFinish"].fillna("No Garage")
df["GarageYrBlt"] = df["GarageYrBlt"].fillna("0")
df["GarageCars"] = df["GarageCars"].fillna("0")
df["GarageArea"] = df["GarageArea"].fillna("0")

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df["BsmtFinType2"] = df["BsmtFinType2"].fillna("No Bsmt")
df["BsmtExposure"] = df["BsmtExposure"].fillna("No Bsmt")
df["BsmtFinType1"] = df["BsmtFinType1"].fillna("No Bsmt")
df["BsmtQual"] = df["BsmtQual"].fillna("No Bsmt")
df["BsmtCond"] = df["BsmtCond"].fillna("No Bsmt")
df["BsmtFinSF1"] = df["BsmtFinSF1"].fillna("No Bsmt")
df["BsmtFinSF2"] = df["BsmtFinSF2"].fillna("No Bsmt")
df["BsmtUnfSF"] = df["BsmtUnfSF"].fillna("No Bsmt")
df["TotalBsmtSF"] = df["TotalBsmtSF"].fillna("No Bsmt")

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df["MasVnrArea"] = df["MasVnrArea"].fillna("0")
df["Electrical"] = df["Electrical"].fillna("None")

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.replace("NA", np.nan, inplace=True)

for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

df.fillna(df.median(numeric_only=True), inplace=True)

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
x = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

categorical_cols = x.select_dtypes(include=["object"]).columns
numerical_cols = x.select_dtypes(exclude=["object"]).columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"),categorical_cols)
])

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ))
])

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

pipeline.fit(x_train, y_train)
        
y_pred = pipeline.predict(x_test)

warnings.filterwarnings("ignore")

In [ ]:
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(r2)
print(mse)

In [ ]:
model = pipeline.named_steps["model"]

feature_names = pipeline.named_steps["preprocessing"].get_feature_names_out()

importance = model.feature_importances_

feature_importance = pd.DataFrame({
    "Feature" : feature_names,
    "Importance" : importance
})

feature_importance = feature_importance.sort_values(by="Importance", ascending = False)
feature_importance.head(10)

In [ ]:
joblib.dump(pipeline, "model.pkl")

In [ ]:
sample = df.drop("SalePrice", axis=1).head(5)
sample.to_csv("sample_input.csv", index =False)

# Developed a model for predicting house prices using XGBoost with a full preprocessing pipeline (handling missing values, encoding categorical variables, and scaling). Achieved an R² score of ~0.90.

# Performed feature importance analysis, identifying key drivers of house prices such as overall quality, living area, garage capacity, and interior features. Found that physical attributes of the property have a stronger impact on pricing than location-based categorical features.